# Сравнение поиска и финальная оценка

Ноутбук сравнивает word TF-IDF и BM25 на одобренных статьях. Запросы разделены на два файла: calibration нужен для выбора метода и порога отказа; holdout — для однократной финальной оценки зафиксированного решения. Не меняйте метод или порог после просмотра holdout. Корпус и файлы разметки должны оставаться локальными.


## 1. Подготовка корпуса

Статьи должны соответствовать схеме из `data/knowledge/README.md`: уникальный ID, ссылка на источник, дата `YYYY-MM-DD`, язык, владелец, лицензия и `status=approved`. BANKING77 не является базой ответов.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CORPUS_PATH = ROOT / 'data' / 'knowledge' / 'articles.jsonl'
CALIBRATION_PATH = ROOT / 'data' / 'knowledge' / 'queries_calibration.jsonl'
HOLDOUT_PATH = ROOT / 'data' / 'knowledge' / 'queries_holdout.jsonl'
retriever = None
bm25_retriever = None
calibration_examples = None
holdout_examples = None


In [ ]:
if not CORPUS_PATH.is_file():
    print('Добавьте одобренные статьи в data/knowledge/articles.jsonl.')
else:
    from src.rag.retriever import BM25Retriever, KnowledgeRetriever
    retriever = KnowledgeRetriever.from_jsonl(CORPUS_PATH)
    bm25_retriever = BM25Retriever.from_jsonl(CORPUS_PATH)
    print(f'Документов: {len({chunk.document_id for chunk in retriever.chunks})}')
    print(f'Фрагментов: {len(retriever.chunks)}')


## 2. Ручной просмотр выдачи

Измените запрос на свой. Сначала сравните, какие документы поставили наверх два ранжировщика. Их численные score лежат в разных шкалах и между собой не сравниваются.


In [ ]:
query = 'card payment reversed'
if retriever is not None:
    tfidf_hits = retriever.search(query, top_k=5)
    for rank, hit in enumerate(tfidf_hits, start=1):
        print(f'{rank}. {hit.title} | {hit.source}')
        print(f'   {hit.score:.4f} ({hit.score_type}) | {hit.text}\n')
    bm25_hits = bm25_retriever.search(query, top_k=5)
    for rank, hit in enumerate(bm25_hits, start=1):
        print(f'{rank}. {hit.title} | {hit.source}')
        print(f'   {hit.score:.4f} ({hit.score_type}) | {hit.text}\n')


## 3. Calibration: сравнение и выбор порога

Составьте два независимых файла вручную. Разделяйте близкие переформулировки одного вопроса между группами целиком, чтобы похожие запросы не попали по разные стороны. В calibration должны быть answerable и unanswerable запросы. В этом разделе сравниваются методы и строятся пороги отказа.


In [ ]:
if retriever is None:
    print('Сначала добавьте корпус и выполните предыдущие ячейки.')
elif not CALIBRATION_PATH.is_file():
    print('Добавьте data/knowledge/queries_calibration.jsonl для сравнения методов и выбора порога.')
else:
    from src.rag.evaluation import evaluate_retrieval, load_queries, sweep_abstention_thresholds
    import pandas as pd
    calibration_examples = load_queries(CALIBRATION_PATH)
    tfidf_calibration = evaluate_retrieval(retriever, calibration_examples, k=5)
    bm25_calibration = evaluate_retrieval(bm25_retriever, calibration_examples, k=5)
    calibration_metrics = pd.DataFrame([
        {'method': 'TF-IDF', 'Recall@5': tfidf_calibration['recall_at_5'], 'MRR': tfidf_calibration['mrr']},
        {'method': 'BM25', 'Recall@5': bm25_calibration['recall_at_5'], 'MRR': bm25_calibration['mrr']},
    ])
    display(calibration_metrics)


In [ ]:
if calibration_examples is not None:
    tfidf_thresholds = pd.DataFrame(
        sweep_abstention_thresholds(retriever, calibration_examples)
    ).assign(method='TF-IDF')
    bm25_thresholds = pd.DataFrame(
        sweep_abstention_thresholds(bm25_retriever, calibration_examples)
    ).assign(method='BM25')
    threshold_results = pd.concat([tfidf_thresholds, bm25_thresholds], ignore_index=True)
    display(threshold_results.sort_values(
        ['method', 'supported_answer_f1', 'coverage'], ascending=[True, False, False]
    ).groupby('method').head(10))


После просмотра calibration вручную зафиксируйте метод и порог ниже. Порог TF-IDF нельзя переносить на BM25. Не выбирайте метод по holdout-результату.


In [ ]:
SELECTED_METHOD = 'tfidf'
SELECTED_THRESHOLD = None

if SELECTED_METHOD not in {'tfidf', 'bm25'}:
    raise ValueError("SELECTED_METHOD должен быть 'tfidf' или 'bm25'.")
print('Выбранный метод:', SELECTED_METHOD)
print('Зафиксированный порог:', SELECTED_THRESHOLD)


## 4. Holdout: однократная финальная оценка

Заполните holdout до выбора метода, не просматривайте его метрики до фиксации метода и порога. После оценки не возвращайтесь к подбору на этом наборе. При изменении ранжировщика или порога понадобится новый независимый holdout.


In [ ]:
if retriever is None:
    print('Сначала добавьте корпус и выполните предыдущие ячейки.')
elif not HOLDOUT_PATH.is_file():
    print('Holdout отсутствует. Финальную метрику пока не считаем.')
elif SELECTED_THRESHOLD is None:
    print('Сначала вручную зафиксируйте порог по calibration в SELECTED_THRESHOLD.')
elif retriever is not None:
    from src.rag.evaluation import evaluate_retrieval, load_queries, sweep_abstention_thresholds
    holdout_examples = load_queries(HOLDOUT_PATH)
    selected_retriever = retriever if SELECTED_METHOD == 'tfidf' else bm25_retriever
    holdout_retrieval = evaluate_retrieval(selected_retriever, holdout_examples, k=5)
    holdout_threshold = sweep_abstention_thresholds(
        selected_retriever, holdout_examples, thresholds=[SELECTED_THRESHOLD]
    )[0]
    print({
        'method': SELECTED_METHOD,
        'threshold': SELECTED_THRESHOLD,
        'recall_at_5': holdout_retrieval['recall_at_5'],
        'mrr': holdout_retrieval['mrr'],
        'supported_answer_precision': holdout_threshold['supported_answer_precision'],
        'supported_answer_recall': holdout_threshold['supported_answer_recall'],
        'supported_answer_f1': holdout_threshold['supported_answer_f1'],
        'coverage': holdout_threshold['coverage'],
        'unsafe_accept_rate': holdout_threshold['unsafe_accept_rate'],
        'unanswerable_false_accept_rate': holdout_threshold['unanswerable_false_accept_rate'],
    })


## Интерпретация

Recall@5 и MRR оценивают ранжирование статей. Coverage и метрики supported answers оценивают принятие/отказ при закреплённом пороге. Score не является вероятностью. Численные результаты появятся только после добавления реального корпуса и экспертной разметки обоих наборов. Подробности протокола и ограничений записаны в Word-пособии и `reports/research.md`.
